In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Path to dataset files:", path)

100%|██████████| 1.21G/1.21G [00:26<00:00, 48.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


In [8]:
# Install required libraries (run once if needed)
# !pip install kagglehub librosa torch torchvision matplotlib

# -----------------------------
# 1. Import libraries
# -----------------------------
import os
import kagglehub
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# -----------------------------
# 2. Download dataset
# -----------------------------
# path = kagglehub.dataset_download(
#     "andradaolteanu/gtzan-dataset-music-genre-classification"
# )

print("Dataset path:", path)

audio_dataset_path = os.path.join(path, "Data", "genres_original")

# -----------------------------
# 3. Convert audio → spectrogram
# -----------------------------
spectrogram_path = "spectrogram_dataset"
os.makedirs(spectrogram_path, exist_ok=True)

genres = os.listdir(audio_dataset_path)

for genre in genres:

    genre_folder = os.path.join(audio_dataset_path, genre)
    save_folder = os.path.join(spectrogram_path, genre)

    os.makedirs(save_folder, exist_ok=True)

    for file in os.listdir(genre_folder):

        if file.endswith(".wav"):

            file_path = os.path.join(genre_folder, file)

            # Load audio
            y, sr = librosa.load(file_path, duration=30)

            # Create Mel spectrogram
            mel = librosa.feature.melspectrogram(y=y, sr=sr)
            mel_db = librosa.power_to_db(mel, ref=np.max)

            # Plot spectrogram
            plt.figure(figsize=(3,3))
            librosa.display.specshow(mel_db, sr=sr)
            plt.axis('off')

            save_path = os.path.join(save_folder, file.replace(".wav",".png"))
            plt.savefig(save_path, bbox_inches='tight', pad_inches=0)
            plt.close()

print("Spectrogram images created!")

# -----------------------------
# 4. Image preprocessing
# -----------------------------
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

# -----------------------------
# 5. Load dataset
# -----------------------------
dataset = datasets.ImageFolder(
    root=spectrogram_path,
    transform=transform
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# -----------------------------
# 6. Load pretrained ResNet50
# -----------------------------
model = models.resnet50(pretrained=True)

# Freeze earlier layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
num_features = model.fc.in_features
num_classes = len(dataset.classes)

model.fc = nn.Linear(num_features, num_classes)

# -----------------------------
# 7. Loss and optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# -----------------------------
# 8. Train model
# -----------------------------
epochs = 2

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {running_loss:.4f}")

# -----------------------------
# 9. Evaluate model
# -----------------------------
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print("Validation Accuracy:", accuracy)

Dataset path: /root/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1


/tmp/ipykernel_782/340100197.py:52: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, duration=30)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


NoBackendError: 